## Load the dataset

This should already be filtered for high quality german sentences

In [1]:
import pandas as pd

dataset = pd.read_json("../data/interim/german_sentences_filtered.jsonl", lines=True)
dataset.head()

,original_sentence,cleaned_sentence,keep,reason
0,Alan Smithee steht als Pseudonym für einen fik...,Alan Smithee steht als Pseudonym für einen fik...,True,good example
1,Von 1968 bis 2000 wurde es von der Directors G...,Von 1968 bis 2000 wurde es von der Directors G...,True,proper structure
2,Alternative Schreibweisen sind unter anderem d...,Alternative Schreibweisen sind unter anderem d...,True,valid sentence
3,Alan Smi Thee und Sumishii Aran gehören so die...,Alan Smi Thee und Sumishii Aran gehören so die...,True,useful structure
4,Regisseur Robert Totten und Hauptdarsteller Ri...,Regisseur Robert Totten und Hauptdarsteller Ri...,True,clear example


# Create Ollama client and Load Env

Load `OLLAMA_API_KEY` variable

In [2]:
from ollama import Client
import os
from dotenv import load_dotenv

load_dotenv() 

client = Client(
            host="https://ollama.com",
            headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
        )

## Create corruption pair

- Using a bigger LLM `GPT-OSS-120b` we create a corrupted pair for each sentence
- The number of issues and the types are also recorded

In [3]:
import json
from src.utils import process_in_batches
from src.data.corruptor import corrupt_sentence
from tqdm import tqdm

all_sentences = dataset["cleaned_sentence"].to_list()[759:]

print(f"Starting to corrupt {len(all_sentences)} rows...")

with tqdm(total=len(all_sentences), desc="Corrupting sentences") as pbar:
    for current_batch in process_in_batches(all_sentences, batch_size=10, verbose=False):
        batch_result = corrupt_sentence(current_batch, client)

        with open("../data/interim/german_sentences_filtered_corrupted.jsonl", "a", encoding="utf-8") as f:
            for line in batch_result:
                f.write(json.dumps(line, ensure_ascii=False) + "\n")
        
        pbar.update(len(current_batch))


Starting to corrupt 2670 rows...


Corrupting sentences: 100%|██████████| 2670/2670 [2:49:55<00:00,  3.82s/it]  


## We filter the data once more for Quality

As the last step `GPT-OSS-120b` goes through the pairs again and removes examples that don't have enough value for training or invalid

In [4]:
df_corrupted = pd.read_json("../data/interim/german_sentences_filtered_corrupted.jsonl", lines=True)
candidates = list(zip(df_corrupted["original"], df_corrupted["corrupted"]))

In [5]:
from tqdm import tqdm
from src.data.quality import validate_corruption_pair
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_SIZE = 3


def process_pair(pair):
    clean, corrupt = pair
    evaluation = validate_corruption_pair(clean, corrupt, client)
    return { "input": corrupt, "label": clean, "verdict": evaluation["verdict"], "reason": evaluation["reason"] }


with tqdm(total=len(candidates), desc="Evaluating pairs") as pbar:
    for i in range(0, len(candidates), BATCH_SIZE):
        batch = candidates[i:i + BATCH_SIZE]
        with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
            futures = {executor.submit(process_pair, pair): pair for pair in batch}
            for future in as_completed(futures):
                evaluated_data = future.result()
                with open("../data/interim/german_sentences_filtered_corrupted_evaluated.jsonl", "a", encoding="utf-8") as f:
                    f.write(json.dumps(evaluated_data, ensure_ascii=False) + "\n")
                pbar.update(1)

Evaluating pairs: 100%|██████████| 3434/3434 [3:25:22<00:00,  3.59s/it]   


# Create the final dataset ready for training

In [ ]:
df_eval = pd.read_json("../data/interim/german_sentences_filtered_corrupted_evaluated.jsonl", lines=True)

df_final = df_eval[df_eval["verdict"] == "KEEP"].copy()
df_final.rename(columns={"input": "corrupted", "label": "original"}, inplace=True)
df_final.to_json("../data/processed/german_sentences_corrupted_final.jsonl", orient="records", lines=True, force_ascii=False)

# Code to fix the discarded rows

we could introduce letter mixups in those sentences that will be 2x discarded

so get discarded -> put into file -> corrupt them again -> quality them again -> mixup letters in discarded ones 

In [1]:
import pandas as pd

df = pd.read_json("../data/interim/train_v9_corrupted_part5_quality.jsonl", lines=True)
df = df[df["verdict"] == "DISCARD"].copy()
print(len(df))
df.drop(columns=["input", "verdict", "reason"], inplace=True)
df.rename(columns={"label": "original"}, inplace=True)
df.head()
df.to_json("../data/interim/discarded.jsonl", orient="records", lines=True, force_ascii=False)

275


In [14]:
import pandas as pd

df = pd.read_json("../data/interim/missing_labels_corrupted_quality.jsonl", lines=True)

# copy the value from label to input where the verdict is DISCARD
df["input"] = df.apply(lambda row: row["label"] if row["verdict"] == "DISCARD" else row["input"], axis=1)

df.head(12)

df.to_json("../data/interim/missing_labels_corrupted_quality_filled.jsonl", orient="records", lines=True, force_ascii=False)

In [ ]:
df2 = pd.read_json("../data/interim/train_v9_corrupted_quality_wip_deduped.jsonl", lines=True)
df2.head(10)

,input,label,verdict,reason
0,Wenn du deine Schulausbildung beendet haben wi...,Wenn du deine Schulausbildung beendet haben wi...,KEEP,"Clear grammatical errors (verb agreement, wron..."
1,"Das bedeutet, es soll möglich werden, Fahrten ...","Das bedeutet, es soll möglich werden, Fahrten ...",KEEP,"Clear case and adjective inflection errors, ea..."
2,"Wir haben gesagt, weil wir es glaubten, aber i...","Wir haben das gesagt, weil wir es glaubten, ab...",KEEP,"Multiple clear grammatical errors (omitted ""da..."
3,"Morgen gehen wir und kaufst ein Spülmaschine, ...","Morgen gehen wir und kaufen eine Spülmaschine,...",KEEP,"Clear verb agreement and article errors, recov..."
4,Donald Trump hatte sich am Freitag besorgt von...,Donald Trump hatte sich am Freitag besorgt übe...,KEEP,"Clear grammatical errors (preposition, case, v..."
5,"Seit dem versteht er auch nicht, was schief ge...","Seitdem versteht er auch nicht, was schief geg...",KEEP,Clear grammatical/spelling errors (incorrect s...
6,Für Rollen in die Bühneninszenierungen von The...,Für seine Rollen in den Bühneninszenierungen v...,KEEP,"Clear case and determiner errors, recoverable ..."
7,B ist ein notwendige Bedingung für A. Dass B g...,B ist eine notwendige Bedingung für A. Dass B ...,KEEP,Clear grammatical errors (article and adjectiv...
8,Nun eingesetzte Marine soll unterdessen ein ve...,Die nun eingesetzte Marine soll unterdessen ei...,KEEP,Clear case and article errors; original meanin...
9,Ich gehst und schau mal auf.,Ich gehe und schau mal nach.,KEEP,"Clear verb‑agreement and preposition errors, n..."


In [4]:
# replace all rows in df2 with the rows from df, where the "label" column in df2 matches the "label" column in df
for idx, row in df.iterrows():
    label_to_replace = row["label"]
    matching_rows = df2[df2["label"] == label_to_replace]
    if not matching_rows.empty:
        df2.loc[matching_rows.index, "input"] = row["input"]

In [5]:
df2.to_json("../data/interim/merged.jsonl", orient="records", lines=True, force_ascii=False)

In [8]:
df = pd.read_json("../data/interim/train_v9_corrupted_quality_wip.jsonl", lines=True)

# print number of duplicates based on the label field
print(df["label"].duplicated().sum())

# remove duplicates based on the label field, keeping the first occurrence
df = df.drop_duplicates(subset=["label"], keep="first")
df.to_json("../data/interim/train_v9_corrupted_quality_wip_deduped.jsonl", orient="records", lines=True, force_ascii=False)

4


In [10]:
df = pd.read_json("../data/interim/train_v9_corrupted_quality_wip_deduped.jsonl", lines=True)
df_train = pd.read_json("../data/processed/train_v8.jsonl", lines=True)

# count how many rows from df_train are missing from df
missing_labels = []
for idx, row in df_train.iterrows():
    label = row["original"]
    if not ((df["label"] == label).any()):
        missing_labels.append(label)

print(f"Number of missing rows: {len(missing_labels)}")

Number of missing rows: 401


In [11]:
missing_labels[0:10]

['Durch elektrolytische Herstellung entsteht explosives Antimon, das beim Ritzen explosionsartig aufglühend und funkensprühend in metallisches Antimon übergeht.',
 'Die Polizei behauptete, dass sie es selbst angezündet haben.',
 'Ich habe geduscht und mir die Zähne geputzt.',
 'Hier sollen aussagenlogische Formeln als Worte über dem Alphabet der logischen Sprache, also über V J wie folgt induktiv definiert werden',
 'Anpassungen für Video-Calls Nachdem WhatsApp schon im April Videoanrufe mit bis zu 8 Teilnehmern möglich gemacht hat, gibt es jetzt weitere Verbesserungen.',
 'Das gilt sowohl für einfache als auch für verknüpfte Aussagen.',
 'Die Stock-Keeping-Units sind für die Lagerhaltung wichtig.',
 'ARM stand ursprünglich für Acorn Risc Machine RISC ist eine Abkürzung aus dem IT Bereich, die eine Bauart von Prozessoren bezeichnet.',
 'LoJack Hersteller von Auffindungssystemen für gestohlene Fahrzeuge, Baumaschinen und Computer ist ein Wortspiel auf die Worte looted US englisch für ge

In [12]:
pd.DataFrame(missing_labels, columns=["original"]).to_json("../data/interim/missing_labels.jsonl", orient="records", lines=True, force_ascii=False)